# 06 Ablation Study

This notebook trains one fixed selected YOLO architecture across the four A/B/C/D training configurations.


## Purpose

Notebook 06 compares training configurations, not architectures. The selected architecture comes from Notebook 05 candidate triage, then stays fixed across all four ablation experiments.

Validation metrics decide the best training configuration. The test split is not used for ablation selection.

To avoid artifact clutter, Ultralytics plots and notebook summary figures are disabled by default. The core output is the compact ablation results, failures, and ranking CSVs.


## Step 1 - Setup

This cell locates the `sign-detection` root, adds it to `sys.path`, and sets Windows/Jupyter environment defaults before any Ultralytics-facing helper is used.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Keep Windows/Jupyter torch startup stable before Ultralytics is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import matplotlib.pyplot as plt  # Used only when optional summary figures are enabled.
import pandas as pd
import yaml
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Walk upward until the sign-detection project root is found."""
    for candidate in [start, *start.parents]:
        if (
            candidate / "configs" / "training_config.yaml"
        ).exists() and candidate.name == "sign-detection":
            return candidate
        if (candidate / "sign-detection" / "configs" / "training_config.yaml").exists():
            return candidate / "sign-detection"
    raise FileNotFoundError("Could not locate the sign-detection project root.")


# Resolve paths from the current working directory so the notebook works from
# either the repo root or the sign-detection folder.
PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Imports happen after PROJECT_ROOT is inserted into sys.path.
from src.training.evaluate_yolo import benchmark_model_latency, rank_ablation_results  # noqa: E402
from src.training.train_yolo import train_ablation_experiments  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Github\smart-factory-safety-monitoring-system\sign-detection


## Step 2 - Load Configuration

Training settings come from `training_config.yaml`. Online augmentation settings are loaded separately so experiments B and D can use them while A and C use the off/minimal values.


In [2]:
def load_yaml(path: Path) -> dict:
    """Load one YAML file using UTF-8 encoding."""
    with path.open("r", encoding="utf-8") as file_handle:
        return yaml.safe_load(file_handle)


training_config = load_yaml(PROJECT_ROOT / "configs" / "training_config.yaml")
augmentation_config = load_yaml(PROJECT_ROOT / "configs" / "augmentation_config.yaml")
class_config = load_yaml(PROJECT_ROOT / "configs" / "class_names.yaml")

# Keep ON/OFF augmentation dictionaries separate so the experiment routing is explicit.
online_aug_args = dict(augmentation_config.get("online_augmentation_on", {}))
offline_aug_args = dict(augmentation_config.get("online_augmentation_off", {}))
if not online_aug_args:
    raise ValueError(
        "online_augmentation_on is missing from configs/augmentation_config.yaml"
    )
if not offline_aug_args:
    raise ValueError(
        "online_augmentation_off is missing from configs/augmentation_config.yaml"
    )

print(f"Classes: {class_config.get('names')}")


Classes: {0: 'M014_Helmet', 1: 'M015_Vest', 2: 'P004_NoThoroughfare', 3: 'W011_Slippery'}


## Step 3 - Resolve Selected Architecture

The notebook first reads the top successful row from `candidate_model_ranking.csv`. If that file is missing or has no successful rows, it falls back to `selected_model` in `training_config.yaml`. It will stop with a clear error instead of guessing.

This matters because ablation must compare training setups only. If the architecture changes between A/B/C/D, the experiment is no longer a clean ablation.


In [3]:
def resolve_selected_model(project_root: Path, config: dict) -> tuple[str, str]:
    """Resolve the fixed architecture for ablation without using test metrics."""
    # Prefer the actual Notebook 05 ranking so ablation follows measured validation results.
    ranking_path = project_root / "reports" / "training" / "candidate_model_ranking.csv"
    if ranking_path.exists():
        ranking_df = pd.read_csv(ranking_path)
        if not ranking_df.empty:
            if "status" in ranking_df.columns:
                # Ignore failed/unsupported candidate rows when choosing the fixed architecture.
                ranking_df = ranking_df.loc[ranking_df["status"].eq("trained")].copy()
            if not ranking_df.empty:
                top_row = ranking_df.iloc[0]
                selected = (
                    top_row.get("weights")
                    or top_row.get("selected_model")
                    or top_row.get("model_name")
                )
                if pd.notna(selected) and str(selected).strip():
                    return str(
                        selected
                    ), f"candidate ranking: {ranking_path.relative_to(project_root)}"

    # Fallback is intentionally explicit; the notebook should not guess a model.
    configured_model = config.get("selected_model")
    if configured_model is not None and str(configured_model).strip():
        return str(configured_model), "configs/training_config.yaml selected_model"

    raise RuntimeError(
        "No selected architecture found. Run Notebook 05 or set selected_model in configs/training_config.yaml."
    )


selected_model, selected_model_source = resolve_selected_model(
    PROJECT_ROOT, training_config
)
print(f"Selected model: {selected_model}")
print(f"Resolved from: {selected_model_source}")


Selected model: yolov9t.pt
Resolved from: candidate ranking: reports\training\candidate_model_ranking.csv


## Step 4 - Define Ablation Experiments

Only the training condition changes across A/B/C/D. Validation and test splits are fixed in the dataset folders created by Notebook 04. Online augmentation is ON only for B and D.

The dataset YAML decides which train images are used. The online augmentation flag decides whether Ultralytics applies runtime augmentation during training. These two concerns stay separate on purpose.


In [4]:
# A/B/C/D use fixed val/test sets from Notebook 04. Only train data and
# online augmentation behavior differ between experiments.
experiment_configs = [
    {
        "experiment": "exp_A_original_only",
        "label": "A_original_only",
        "dataset_yaml": PROJECT_ROOT / "data_exp_A_original_only.yaml",
        # A: baseline data only; online augmentation OFF.
        # C: offline augmented train data is included, online augmentation OFF.
        "use_online_augmentation": False,
        "meaning": "original train only; online augmentation OFF",
    },
    {
        "experiment": "exp_B_online_aug",
        "label": "B_online_aug",
        "dataset_yaml": PROJECT_ROOT / "data_exp_B_online_aug.yaml",
        # B: same original train data as A, but online augmentation ON.
        # D: offline augmented train data plus online augmentation ON.
        "use_online_augmentation": True,
        "meaning": "original train only; online augmentation ON",
    },
    {
        "experiment": "exp_C_offline_aug",
        "label": "C_offline_aug",
        "dataset_yaml": PROJECT_ROOT / "data_exp_C_offline_aug.yaml",
        "use_online_augmentation": False,
        "meaning": "original + offline augmented train; online augmentation OFF",
    },
    {
        "experiment": "exp_D_full_pipeline",
        "label": "D_full_pipeline",
        "dataset_yaml": PROJECT_ROOT / "data_exp_D_full_pipeline.yaml",
        "use_online_augmentation": True,
        "meaning": "original + offline augmented train; online augmentation ON",
    },
]

# Fail before training if Notebook 04 did not create every dataset YAML.
missing_yamls = [
    str(item["dataset_yaml"])
    for item in experiment_configs
    if not Path(item["dataset_yaml"]).exists()
]
if missing_yamls:
    raise FileNotFoundError(
        f"Missing ablation dataset YAML files. Run Notebook 04 first: {missing_yamls}"
    )

experiment_table = pd.DataFrame(
    [
        {
            "experiment": item["experiment"],
            "dataset_yaml": Path(item["dataset_yaml"]).name,
            "online_augmentation": "ON" if item["use_online_augmentation"] else "OFF",
            "meaning": item["meaning"],
        }
        for item in experiment_configs
    ]
)
display(experiment_table)


,experiment,dataset_yaml,online_augmentation,meaning
0,exp_A_original_only,data_exp_A_original_only.yaml,OFF,original train only; online augmentation OFF
1,exp_B_online_aug,data_exp_B_online_aug.yaml,ON,original train only; online augmentation ON
2,exp_C_offline_aug,data_exp_C_offline_aug.yaml,OFF,original + offline augmented train; online aug...
3,exp_D_full_pipeline,data_exp_D_full_pipeline.yaml,ON,original + offline augmented train; online aug...


## Step 5 - Build Shared Training Arguments

The selected architecture is fixed, so all experiments share the same base training settings. The helper attaches online augmentation ON values only for B/D and OFF values for A/C.

The artifact controls are deliberately conservative. A full ablation run already creates four YOLO run folders, so per-run plots and notebook figures stay disabled unless you explicitly need them.


In [5]:
# Set DRY_RUN=True to verify paths and experiment definitions without launching training.
DRY_RUN = False

# Keep existing ablation runs unless you intentionally want to replace them.
OVERWRITE_EXISTING_RUNS = False

# Validation metrics already come from training. Latency is compact and useful for ranking.
RUN_LATENCY_BENCHMARK = True
MAX_BENCHMARK_IMAGES = 20

# Artifact controls. Keep these off for normal experimentation to avoid many per-run images.
SAVE_ULTRALYTICS_PLOTS = False
SAVE_SUMMARY_FIGURES = False

# Epoch fallback keeps the notebook usable even if configs evolve over time.
ablation_epochs = int(
    training_config.get(
        "ablation_epochs",
        training_config.get(
            "candidate_epochs",
            training_config.get("final_epochs", training_config.get("epochs", 75)),
        ),
    )
)
# These base settings are identical for A/B/C/D; augmentation is attached later
# by train_ablation_experiments based on each experiment config.
base_train_args = {
    "epochs": ablation_epochs,
    "imgsz": int(training_config.get("imgsz", 640)),
    "batch": int(training_config.get("batch", 16)),
    "device": training_config.get("device", 0),
    "workers": int(training_config.get("workers", 8)),
    "patience": int(training_config.get("patience", 30)),
    "seed": int(training_config.get("seed", 42)),
    "plots": SAVE_ULTRALYTICS_PLOTS,
    "dry_run": DRY_RUN,
    "overwrite": OVERWRITE_EXISTING_RUNS,
}

# Optional config key: if absent, let Ultralytics choose its default AMP behavior.
if "amp" in training_config:
    base_train_args["amp"] = bool(training_config["amp"])

# Display the exact base settings before any long training starts.
settings_preview = pd.DataFrame(
    [{"setting": key, "value": value} for key, value in base_train_args.items()]
)
display(settings_preview)


,setting,value
0,epochs,75
1,imgsz,640
2,batch,16
3,device,0
4,workers,8
5,patience,30
6,seed,42
7,plots,False
8,dry_run,False
9,overwrite,False


## Step 6 - Verify Validation Image Source for Latency

Latency benchmarking uses validation images only. Since validation is fixed across all ablation datasets, using the validation folder from Experiment A is enough.


In [6]:
def validation_images_from_dataset_yaml(data_yaml: Path) -> Path:
    """Resolve the validation image folder from one Ultralytics dataset YAML."""
    # Dataset YAML paths are repo-relative, so resolve them from sign-detection root.
    config = load_yaml(data_yaml)
    dataset_root = (data_yaml.parent / config["path"]).resolve()
    return dataset_root / config["val"]


# Validation is identical across all experiments; using A avoids repeated path parsing.
val_images_dir = validation_images_from_dataset_yaml(
    Path(experiment_configs[0]["dataset_yaml"])
)
if not val_images_dir.exists():
    raise FileNotFoundError(f"Validation image folder not found: {val_images_dir}")
print(f"Validation images for latency benchmark: {val_images_dir}")


Validation images for latency benchmark: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\experiments\exp_A_original_only\val\images


## Step 7 - Train Ablation Experiments

This is the training cell. Each row trains the same selected architecture with one A/B/C/D dataset and its matching online augmentation setting. Failed experiments are recorded and do not stop the remaining runs.


In [7]:
ablation_runs_dir = PROJECT_ROOT / "runs" / "ablation"
reports_training_dir = PROJECT_ROOT / "reports" / "training"
reports_training_dir.mkdir(parents=True, exist_ok=True)

# The helper records failed experiments and continues, so partial results are still visible.
# It also preserves existing run folders by adding a suffix unless overwrite=True.
ablation_results = train_ablation_experiments(
    selected_model=selected_model,
    experiment_configs=experiment_configs,
    output_dir=ablation_runs_dir,
    base_train_args=base_train_args,
    # B/D receive online_aug_args; A/C receive offline_aug_args.
    online_aug_args=online_aug_args,
    offline_aug_args=offline_aug_args,
    continue_on_error=True,
)

display(ablation_results)


New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Github\smart-factory-safety-monitoring-system\sign-detection\runs\ablation\_data_exp_A_original_only_resolved.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_rat

,experiment,dataset_yaml,selected_model,use_online_augmentation,run_name,status,resolved_data_yaml,run_dir,precision,recall,...,fitness,training_time,best_weights_path,last_weights_path,model_size_mb,fps,avg_latency_ms,num_benchmark_images,error_message,notes
0,exp_A_original_only,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,False,A_original_only_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.97640,0.90934,...,None,374.297,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,None,None,None,,
1,exp_B_online_aug,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,True,B_online_aug_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.93535,0.97024,...,None,362.319,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,None,None,None,,
2,exp_C_offline_aug,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,False,C_offline_aug_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96042,0.89356,...,None,427.487,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.425424,None,None,None,,
3,exp_D_full_pipeline,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,True,D_full_pipeline_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96209,0.95704,...,None,553.706,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,None,None,None,,


## Step 9 - Rank and Save Reports

Successful experiments are ranked by validation recall, mAP50, mAP50-95, FPS, then smaller model size. In dry-run mode the notebook displays results but does not overwrite canonical ablation report CSVs.

This keeps quick wiring checks from accidentally replacing real experiment results.


In [8]:
updated_rows = []
for row in ablation_results.to_dict("records"):
    # Failed or dry-run rows stay in the report but are not benchmarked.
    if row.get("status") != "trained":
        updated_rows.append(row)
        continue

    # Use best.pt from each successful run; if a run lacks weights, skip cleanly.
    best_weights_path = Path(str(row.get("best_weights_path", "")))
    if RUN_LATENCY_BENCHMARK and best_weights_path.exists():
        # Latency is measured on a small deterministic validation subset.
        latency_metrics = benchmark_model_latency(
            weights_path=best_weights_path,
            sample_images_dir=val_images_dir,
            imgsz=int(base_train_args["imgsz"]),
            device=base_train_args["device"],
            max_images=MAX_BENCHMARK_IMAGES,
        )
        row.update(latency_metrics)
    updated_rows.append(row)

ablation_results = pd.DataFrame(updated_rows)
display(ablation_results)


,experiment,dataset_yaml,selected_model,use_online_augmentation,run_name,status,resolved_data_yaml,run_dir,precision,recall,...,training_time,best_weights_path,last_weights_path,model_size_mb,fps,avg_latency_ms,num_benchmark_images,error_message,notes,latency_warning
0,exp_A_original_only,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,False,A_original_only_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.97640,0.90934,...,374.297,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,21.515199,46.478770,20,,,
1,exp_B_online_aug,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,True,B_online_aug_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.93535,0.97024,...,362.319,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,23.727410,42.145350,20,,,
2,exp_C_offline_aug,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,False,C_offline_aug_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96042,0.89356,...,427.487,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.425424,24.047343,41.584635,20,,,
3,exp_D_full_pipeline,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,True,D_full_pipeline_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96209,0.95704,...,553.706,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,23.943488,41.765010,20,,,


## Step 9 - Rank and Save Reports

Successful experiments are ranked by validation recall, mAP50, mAP50-95, FPS, then smaller model size. In dry-run mode the notebook displays results but does not overwrite canonical ablation report CSVs.


In [9]:
ablation_ranking = rank_ablation_results(ablation_results)

results_path = reports_training_dir / "ablation_results.csv"
failures_path = reports_training_dir / "ablation_failures.csv"
ranking_path = reports_training_dir / "ablation_ranking.csv"

# Keep a stable failures file schema even when every experiment succeeds.
failure_columns = [
    "experiment",
    "dataset_yaml",
    "selected_model",
    "status",
    "error_message",
]
failures_df = ablation_results.loc[
    ablation_results["status"].eq("failed"), failure_columns
].copy()
if failures_df.empty:
    failures_df = pd.DataFrame(columns=failure_columns)

# Dry-run should be safe to execute repeatedly without overwriting real training results.
if DRY_RUN:
    print("DRY_RUN=True: canonical ablation CSV reports were not overwritten.")
else:
    # Keep persistent output compact: results, failures, and ranking CSVs only.
    ablation_results.to_csv(results_path, index=False)
    failures_df.to_csv(failures_path, index=False)
    ablation_ranking.to_csv(ranking_path, index=False)
    print(f"Saved results: {results_path.relative_to(PROJECT_ROOT)}")
    print(f"Saved failures: {failures_path.relative_to(PROJECT_ROOT)}")
    print(f"Saved ranking: {ranking_path.relative_to(PROJECT_ROOT)}")

display(ablation_ranking)


Saved results: reports\training\ablation_results.csv
Saved failures: reports\training\ablation_failures.csv
Saved ranking: reports\training\ablation_ranking.csv


,rank,experiment,dataset_yaml,selected_model,use_online_augmentation,run_name,status,resolved_data_yaml,run_dir,precision,...,training_time,best_weights_path,last_weights_path,model_size_mb,fps,avg_latency_ms,num_benchmark_images,error_message,notes,latency_warning
0,1,exp_B_online_aug,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,True,B_online_aug_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.93535,...,362.319,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,23.727410,42.145350,20,,,
1,2,exp_D_full_pipeline,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,True,D_full_pipeline_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96209,...,553.706,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,23.943488,41.765010,20,,,
2,3,exp_A_original_only,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,False,A_original_only_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.97640,...,374.297,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.427133,21.515199,46.478770,20,,,
3,4,exp_C_offline_aug,C:\Github\smart-factory-safety-monitoring-syst...,yolov9t.pt,False,C_offline_aug_yolov9t,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96042,...,427.487,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.425424,24.047343,41.584635,20,,,


## Step 10 - Optional Summary Figures

Summary figures are disabled by default to avoid cluttering `reports/figures/`. Turn `SAVE_SUMMARY_FIGURES=True` only when you need quick visuals for a report.


In [10]:
if not SAVE_SUMMARY_FIGURES:
    print(
        "Summary figures are disabled. Set SAVE_SUMMARY_FIGURES=True in Step 5 if you need them."
    )
elif ablation_ranking.empty:
    print("No successful ablation rows available for plotting.")
else:
    # Create figure files only when explicitly requested in Step 5.
    figures_dir = PROJECT_ROOT / "reports" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)
    plot_specs = [
        ("recall", "Validation Recall", figures_dir / "ablation_recall.png"),
        ("map50", "Validation mAP50", figures_dir / "ablation_map50.png"),
        (
            "avg_latency_ms",
            "Average Latency (ms)",
            figures_dir / "ablation_latency.png",
        ),
    ]
    for column, title, output_path in plot_specs:
        if (
            column not in ablation_ranking.columns
            or ablation_ranking[column].dropna().empty
        ):
            print(f"Skipped {title}: no {column} values available.")
            continue
        plt.figure(figsize=(8, 4))
        plt.bar(ablation_ranking["experiment"], ablation_ranking[column])
        plt.title(title)
        plt.xlabel("Experiment")
        plt.ylabel(column)
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.savefig(output_path, dpi=150)
        plt.show()
        print(f"Saved figure: {output_path.relative_to(PROJECT_ROOT)}")


Summary figures are disabled. Set SAVE_SUMMARY_FIGURES=True in Step 5 if you need them.


## Step 11 - Notebook 07 Checklist

Before moving to Notebook 07:

- The selected architecture was resolved from Notebook 05 ranking or config.
- All four ablation dataset YAML files were checked.
- Online augmentation was ON only for B and D.
- Online augmentation was OFF/minimal for A and C.
- The ranking was based on validation metrics only.
- The test split was not used for ablation selection.
- The top successful row in `ablation_ranking.csv` can be used for final training/evaluation.
